# ICS40125 - Laboratorio N°05

Exploración visual del catálogo de Netflix con matplotlib y seaborn.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

df = pd.read_csv('https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/netflix_titles.csv')

# preparacion basica que usaremos en varios graficos
df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), format='%B %d, %Y', errors='coerce')
df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month
df['duration_min'] = df['duration'].str.extract(r'(\d+)\s*min').astype(float)
df.head()

## Parte 1: Exploración visual básica

In [ ]:
# 1. Distribucion de tipos de contenido
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='type')
plt.title('Cantidad de titulos por tipo')
plt.show()
# Predomina claramente Movie sobre TV Show.

In [ ]:
# 2. Histograma de anios de lanzamiento
plt.figure(figsize=(10,4))
sns.histplot(df['release_year'], bins=30)
plt.title('Distribucion de release_year')
plt.show()
# La mayoria del contenido es reciente (concentrado despues de 2010).

In [ ]:
# 3. Proporcion de clasificaciones por edad
plt.figure(figsize=(10,4))
orden = df['rating'].value_counts().index
sns.countplot(data=df, x='rating', order=orden)
plt.xticks(rotation=45)
plt.title('Distribucion de rating')
plt.show()
# TV-MA y TV-14 dominan: Netflix apunta mas a publico adulto/adolescente.

## Parte 2: Tendencias y evolución en el tiempo

In [ ]:
# 4. Titulos agregados por anio
por_anio = df['year_added'].value_counts().sort_index()
plt.figure(figsize=(10,4))
plt.plot(por_anio.index, por_anio.values, marker='o')
plt.title('Titulos agregados por anio')
plt.xlabel('anio'); plt.ylabel('cantidad')
plt.show()
# El catalogo crecio fuerte entre 2016 y 2019.

In [ ]:
# 5. Heatmap de lanzamientos por anio y mes
tabla = df.pivot_table(index='month_added', columns='year_added',
                       values='show_id', aggfunc='count', fill_value=0)
plt.figure(figsize=(12,6))
sns.heatmap(tabla, cmap='YlGnBu')
plt.title('Titulos agregados por anio y mes')
plt.show()

In [ ]:
# 6. Duracion de peliculas por genero principal
peliculas = df[df['type'] == 'Movie'].copy()
peliculas['genero_principal'] = peliculas['listed_in'].str.split(', ').str[0]
top = peliculas['genero_principal'].value_counts().head(8).index
plt.figure(figsize=(12,6))
sns.boxplot(data=peliculas[peliculas['genero_principal'].isin(top)],
            x='duration_min', y='genero_principal')
plt.title('Duracion (min) por genero principal')
plt.show()

## Parte 3: Comparaciones y relaciones

In [ ]:
# 7. Top 10 paises con mas producciones
top_paises = df['country'].value_counts().head(10)
plt.figure(figsize=(10,5))
sns.barplot(x=top_paises.values, y=top_paises.index)
plt.title('Top 10 paises con mas titulos')
plt.xlabel('cantidad')
plt.show()
# Estados Unidos e India lideran.

In [ ]:
# 8. Peliculas vs Series por genero (barras apiladas)
dg = df.assign(genre=df['listed_in'].str.split(', ')).explode('genre')
tabla_gt = dg.groupby(['genre', 'type']).size().unstack(fill_value=0)
tabla_gt = tabla_gt.loc[tabla_gt.sum(axis=1).sort_values(ascending=False).head(10).index]
tabla_gt.plot(kind='barh', stacked=True, figsize=(10,6))
plt.title('Titulos por genero (Movie vs TV Show)')
plt.show()

In [ ]:
# 9. Relacion entre duracion y anio de lanzamiento
plt.figure(figsize=(10,5))
sns.scatterplot(data=peliculas, x='release_year', y='duration_min', alpha=0.4)
plt.title('release_year vs duracion (min)')
plt.show()
# La duracion se mantiene relativamente estable; hay algunos outliers muy largos.

## Desafío Final

In [ ]:
# Combinaciones mas frecuentes de genero + rating
dg = df.assign(genre=df['listed_in'].str.split(', ')).explode('genre')
top_gen = dg['genre'].value_counts().head(10).index
top_rat = dg['rating'].value_counts().head(8).index
sub = dg[dg['genre'].isin(top_gen) & dg['rating'].isin(top_rat)]
matriz = sub.pivot_table(index='genre', columns='rating',
                         values='show_id', aggfunc='count', fill_value=0)
plt.figure(figsize=(12,7))
sns.heatmap(matriz, annot=True, fmt='d', cmap='Reds')
plt.title('Frecuencia genero vs rating')
plt.show()
# Los generos internacionales y dramas se asocian a TV-MA/TV-14 (publico adulto).